# ТЕМА 7. RAG-системи

У цьому занятті ми розглянемо базову RAG-систему та її три ключові компоненти: індексацію, отримання (retrieval) та генерацію. RAG дозволяє мовним моделям використовувати зовнішні джерела знань для генерації більш точних відповідей.

### Встановлення пакетів

In [ ]:
!pip install -q "langchain>=0.3.0,<1.0.0" "langsmith>=0.1.99"
!pip install -q beautifulsoup4 chromadb langchain-community langchain-openai langchainhub langchain-text-splitters

### Налаштування API ключів

In [7]:
import os

os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY_HERE"

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "YOUR_LANGSMITH_API_KEY_HERE"
os.environ["LANGSMITH_PROJECT"] = "YOUR_PROJECT_NAME_HERE"

os.environ['USER_AGENT'] = "myagent"

### Фаза індексації

Завантаження документа з веб-сторінки

In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()
print(f"Завантажено документів: {len(docs)}")

Завантажено документів: 1


Розбиття документа на частини

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)
print(f"Створено фрагментів: {len(splits)}")

Створено фрагментів: 63


Створення векторного сховища з ембедингами

In [8]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings()
)

print("Векторне сховище створено")

Векторне сховище створено


### Отримання (Retrieval)

Пошук релевантних документів за запитом

In [9]:
retriever = vectorstore.as_retriever()

question = "What is Task Decomposition?"
docs = retriever.invoke(question)

print(f"Знайдено документів: {len(docs)}")
print(f"\nПерший документ:\n{docs[0].page_content[:500]}...")

Знайдено документів: 4

Перший документ:
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an ...


### Генерація

Створення промпта для RAG-системи

In [10]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

Побудова та тестування RAG-ланцюга

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke("What is Task Decomposition?")
print("Відповідь:", response)

Відповідь: Task Decomposition is a technique used to break down complex tasks into smaller and simpler steps. This approach helps agents plan ahead and tackle difficult tasks more effectively. Different methods, such as using simple prompts or relying on external planners, can be employed for task decomposition.
